# Solver Demo: compare any order with iHub

Select an Order ID and run **Fast** and **Best** on the same order, catalogue and constraints. Compare their validated results against iHub's historical recommendation.

1. Load the sample orders and packing rules.
2. Inspect a single-order packing decision and physical 3D placements.
3. Refresh the full benchmark **explicitly** when the solver changes.
4. Inspect the scorecard and run live order comparisons.

**Important:** iHub is a reference, never a solver input. Saved benchmark results are historical until the refresh cell is executed. Individual worked examples are evaluated live; do not assume earlier carton choices still apply.

In [ ]:
import copy
import json
import math
import statistics
from contextlib import redirect_stdout
from html import escape as html_escape
from io import StringIO
from pathlib import Path
import sys

from IPython.display import HTML, Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'notebooks'))

from bin_packing_3d import (
    display_table, make_table, result_tables, solve_order, visualize_result,
)

def show_details(summary, content):
    body = content._repr_html_() if hasattr(content, '_repr_html_') else f'<pre>{html_escape(str(content))}</pre>'
    display(HTML(f'<details><summary><strong>{html_escape(summary)}</strong></summary>{body}</details>'))

## 1. Load the carton catalogue and constraints

The solver receives candidate cartons and packing rules as inputs. The following cells show the actual catalogue and fill policy from the sample request.

In [ ]:
DATA_PATH = ROOT / 'data' / 'raw' / 'data_samples_v2.json'
with DATA_PATH.open(encoding='utf-8') as file:
    ihub_records = {record['input']['OrderId']: record for record in json.load(file)}

catalogue_source = next(iter(ihub_records.values()))['input']
boxes = [dict(box) for box in catalogue_source['Bins']['BinsList']]
box_rows = [
    {
        'Code': box['Code'], 'Length (mm)': box['Length'],
        'Width (mm)': box['Width'], 'Height (mm)': box['Height'],
        'Max weight (kg)': box['MaxWeight'],
    }
    for box in boxes
]
show_details('Show the six candidate cartons', make_table(box_rows))

### Rules visible at the call boundary

The values below come from the same iHub request. The important reading is: **up to six physical items may use 100% of usable volume; seven or more are capped at 70%; and every carton loses 6 mm of usable height.** Logging and visualization are explicit options and do not change the solution.

In [ ]:
source_parameters = catalogue_source['Bins']['Parameters']
source_buffer = source_parameters['BinBuffer']
high_item_count_threshold = source_parameters['BinMaxFillCheckMinItemQty']
high_item_count_max_fill_pct = source_parameters['BinMaxFillPct']
max_fill_pct = 100
bin_buffer = {
    'length': source_buffer['Length'],
    'width': source_buffer['Width'],
    'height': source_buffer['Height'],
}
display_table([
    {'Rule': 'Default mode', 'Configured value': 'fast'},
    {'Rule': 'Full-fill item threshold', 'Configured value': high_item_count_threshold},
    {'Rule': 'Maximum fill above threshold', 'Configured value': f'{high_item_count_max_fill_pct}%'},
    {'Rule': 'Maximum fill at/below threshold', 'Configured value': f'{max_fill_pct}%'},
    {'Rule': 'Carton clearance (L × W × H)', 'Configured value': f"{bin_buffer['length']} × {bin_buffer['width']} × {bin_buffer['height']} mm"},
])

## 2. How a packing decision is made

Both modes enforce the same weight, clearance, allowed rotation, fill and non-overlapping 3D placement constraints. **Fast is not just a single first-fit pass:** it first constructs a greedy valid plan and can then attempt up to three promising fewer-carton alternatives. **Best** starts from Fast's valid result and searches for a better packing with an additional optimization budget.

| Step | What happens | Why it matters |
| --- | --- | --- |
| Prepare | Expand quantities into physical units and allowed orientations | Every unit and rotation rule counts |
| Screen | Check dimensions, weight and usable volume | Reject impossible candidates early |
| Fast placement | Construct a valid greedy 3D arrangement | Physical fit cannot be inferred from volume alone |
| Fast improvement | Try up to three promising fewer-carton candidates | Fast may improve on its initial greedy result |
| Best search | Use the Fast result as fallback and search further | Spend more time only when quality matters |
| Validate | Independently check every returned placement | No invalid arrangement is accepted |

## 3. Single-order walkthrough: order 80

Run the current Fast implementation and inspect the selected carton(s), independent validation and 3D placement. The result is calculated live; carton selection is **not** assumed in advance.

In [ ]:
source_order_80 = ihub_records[80]['input']
items_order_80 = [dict(item) for item in source_order_80['Items']['ItemsList']]
order_80 = {
    'OrderId': source_order_80['OrderId'],
    'OrderNo': source_order_80['OrderNo'],
    'Items': items_order_80,
}
item_rows_80 = [
    {
        **item, 'Length (mm)': item['Length'], 'Width (mm)': item['Width'],
        'Height (mm)': item['Height'], 'Weight (kg)': item['Weight'],
    }
    for item in items_order_80
]
show_details('Show order 80 source items', make_table(item_rows_80, [
    'Code', 'Length (mm)', 'Width (mm)', 'Height (mm)', 'Weight (kg)',
    'Quantity', 'VerticalRotation', 'UOM',
]))

In [ ]:
order_80_log = StringIO()
with redirect_stdout(order_80_log):
    order_80_result = solve_order(
        order_80,
        boxes,
        mode='fast',
        high_item_count_threshold=high_item_count_threshold,
        high_item_count_max_fill_pct=high_item_count_max_fill_pct,
        max_fill_pct=max_fill_pct,
        bin_buffer=bin_buffer,
        dimension_unit='mm',
        logs=True,
    )

order_80_summary, order_80_placements = result_tables(order_80_result)
display(Markdown(f'### Live Fast decision: {order_80_result.metrics.box_count} carton(s), validated: {order_80_result.validation.valid}'))
display(order_80_summary)
visualize_result(order_80_result)
show_details('Show all item placements', order_80_placements)
show_details('Show the full carton-screening and 3D-search trace', order_80_log.getvalue())

## 4. A constraint must change the answer

Order 285 illustrates why upright-only restrictions matter. The next cells solve the original order and a counterfactual that permits additional rotations. Compare the **live** results rather than assuming a particular carton remains optimal after solver changes.

In [ ]:
source_order_285 = ihub_records[285]['input']
items_order_285 = [dict(item) for item in source_order_285['Items']['ItemsList']]
order_285 = {
    'OrderId': source_order_285['OrderId'],
    'OrderNo': source_order_285['OrderNo'],
    'Items': items_order_285,
}
item_rows_285 = [
    {
        **item, 'Length (mm)': item['Length'], 'Width (mm)': item['Width'],
        'Height (mm)': item['Height'], 'Weight (kg)': item['Weight'],
    }
    for item in items_order_285
]
show_details('Show order 285 source items', make_table(item_rows_285, [
    'Code', 'Length (mm)', 'Width (mm)', 'Height (mm)', 'Weight (kg)',
    'Quantity', 'VerticalRotation', 'UOM',
]))

In [ ]:
order_285_result = solve_order(
    order_285,
    boxes,
    mode='best',
    high_item_count_threshold=high_item_count_threshold,
    high_item_count_max_fill_pct=high_item_count_max_fill_pct,
    max_fill_pct=max_fill_pct,
    bin_buffer=bin_buffer,
    dimension_unit='mm',
)
order_285_summary, order_285_placements = result_tables(order_285_result)
display(Markdown('### Valid decision: preserve upright orientation'))
display(order_285_summary)
show_details('Show validated item orientations and coordinates', order_285_placements)

### Counterfactual: relaxing a rotation restriction changes the problem

The counterfactual permits rotations that the source order disallows. Even if it produces a smaller carton, **that is not a valid recommendation for the original order**. Check the actual packed orientations and carton choices below.

In [ ]:
items_285_ignore_upright = [
    {**item, 'VerticalRotation': 1} for item in items_order_285
]
order_285_ignore_upright = {**order_285, 'Items': items_285_ignore_upright}
order_285_ignore_upright_result = solve_order(
    order_285_ignore_upright,
    boxes,
    mode='best',
    high_item_count_threshold=high_item_count_threshold,
    high_item_count_max_fill_pct=high_item_count_max_fill_pct,
    max_fill_pct=max_fill_pct,
    bin_buffer=bin_buffer,
    dimension_unit='mm',
)

display_table([
    {
        'Scenario': 'Respect source rotation rules',
        'Box type': order_285_result.packed_boxes[0].box_code,
        'Item qty': order_285_result.metrics.packed_item_count,
        'Usable fill (%)': round(order_285_result.metrics.boxes[0].usable_utilization_pct, 2),
        'Valid for source order': 'YES',
    },
    {
        'Scenario': 'Ignore upright-only restrictions',
        'Box type': order_285_ignore_upright_result.packed_boxes[0].box_code,
        'Item qty': order_285_ignore_upright_result.metrics.packed_item_count,
        'Usable fill (%)': round(order_285_ignore_upright_result.metrics.boxes[0].usable_utilization_pct, 2),
        'Valid for source order': 'NO — 5 upright-only units are laid down',
    },
])

source_item_by_code_285 = {item['Code']: item for item in items_order_285}
ignored_rotation_rows = []
for placement in order_285_ignore_upright_result.placements:
    source_item = source_item_by_code_285[placement.item_code]
    if source_item['VerticalRotation'] == 0:
        ignored_rotation_rows.append({
            'Item': placement.item_instance_id,
            'Required source orientation (mm)': f"{source_item['Length']:g} × {source_item['Width']:g} × {source_item['Height']:g}",
            'Counterfactual packed orientation (mm)': f"{placement.orientation.length:g} × {placement.orientation.width:g} × {placement.orientation.height:g}",
            'Rule violation': 'Upright-only item laid down',
        })
display_table(ignored_rotation_rows)

## 5. Overview: which orders deserve attention?

The scorecard reads a saved benchmark CSV. **After changing Fast, refresh the CSV with the explicit cell below before interpreting the figures.** The refresh runs both strategies for all orders and can take time; it is intentionally not triggered on notebook startup.

In [ ]:
# Explicit benchmark refresh — run this ONLY when you want to recompute all 2,000 orders.
# This executes the current Fast and Best implementations against the same v2 records.
# The saved CSV is overwritten locally; review and commit it with the notebook.
REFRESH_BENCHMARK = False  # Set True and run this cell after solver changes.
if REFRESH_BENCHMARK:
    import csv
    from benchmark_solver_modes import benchmark, print_summary
    refreshed_rows = benchmark(list(ihub_records.values()))
    assert len(refreshed_rows) == 2000
    target = ROOT / 'notebooks' / 'artifacts' / 'benchmark_2000_best_vs_ihub.csv'
    with target.open('w', newline='', encoding='utf-8') as output:
        writer = csv.DictWriter(output, fieldnames=list(refreshed_rows[0]))
        writer.writeheader()
        writer.writerows(refreshed_rows)
    print_summary(refreshed_rows)
    print(f'Refreshed {target}; rerun the scorecard cells below.')
else:
    print('Benchmark refresh skipped. Saved scorecard may reflect an older solver.')


In [ ]:
# Load the preserved benchmark instead of recalculating all 2,000 orders.
import csv

BENCHMARK_CSV = ROOT / 'notebooks' / 'artifacts' / 'benchmark_2000_best_vs_ihub.csv'
with BENCHMARK_CSV.open(newline='', encoding='utf-8') as file:
    benchmark_rows = list(csv.DictReader(file))

# CSV values are strings. Convert only the fields used in the scorecard.
number_fields = ('ihub_cartons', 'fast_cartons', 'best_cartons',
                 'ihub_external_volume_mm3', 'fast_external_volume_mm3', 'best_external_volume_mm3',
                 'ihub_largest_carton_volume_mm3', 'fast_largest_carton_volume_mm3',
                 'best_largest_carton_volume_mm3', 'ihub_latency_ms',
                 'fast_runtime_ms', 'best_runtime_ms')
for row in benchmark_rows:
    for field in number_fields:
        row[field] = float(row[field])
    for field in ('best_optimality_proven', 'ihub_box9_over_fill_cap'):
        row[field] = row[field].lower() == 'true'
order_count = len(benchmark_rows)
assert order_count == 2000, f'Expected 2,000 benchmark orders, found {order_count}'

def carton_comparison(mode):
    fewer = sum(row[f'{mode}_cartons'] < row['ihub_cartons'] for row in benchmark_rows)
    same = sum(row[f'{mode}_cartons'] == row['ihub_cartons'] for row in benchmark_rows)
    return fewer, same, order_count - fewer - same

def objective(row, mode):
    return (
        row[f'{mode}_cartons'],
        row[f'{mode}_largest_carton_volume_mm3'],
        row[f'{mode}_external_volume_mm3'],
    )

catalogue_volume = {
    box['Code']: box['Length'] * box['Width'] * box['Height']
    for box in boxes
}

def ihub_objective(row):
    codes = row['ihub_boxes'].split(',')
    return (
        row['ihub_cartons'],
        max(catalogue_volume[code] for code in codes),
        row['ihub_external_volume_mm3'],
    )

def percentile_95(values):
    ordered = sorted(values)
    return ordered[math.ceil(0.95 * len(ordered)) - 1]

fast_counts = carton_comparison('fast')
best_counts = carton_comparison('best')
fast_times = [row['fast_runtime_ms'] for row in benchmark_rows]
best_times = [row['best_runtime_ms'] for row in benchmark_rows]
ihub_times = [row['ihub_latency_ms'] for row in benchmark_rows]
best_improved = sum(objective(row, 'best') < objective(row, 'fast') for row in benchmark_rows)
best_tied = sum(objective(row, 'best') == objective(row, 'fast') for row in benchmark_rows)
best_worse = order_count - best_improved - best_tied
best_proven = sum(row['best_optimality_proven'] for row in benchmark_rows)
box9_over_cap = sum(row['ihub_box9_over_fill_cap'] for row in benchmark_rows)
best_more_rows = [row for row in benchmark_rows if row['best_cartons'] > row['ihub_cartons']]
best_more_with_ihub_cap_failure = sum(row['ihub_box9_over_fill_cap'] for row in best_more_rows)
best_more_with_ihub_cap_pass = len(best_more_rows) - best_more_with_ihub_cap_failure
valid_ihub_rows = [row for row in benchmark_rows if not row['ihub_box9_over_fill_cap']]
valid_best_fewer = sum(row['best_cartons'] < row['ihub_cartons'] for row in valid_ihub_rows)
valid_best_same = sum(row['best_cartons'] == row['ihub_cartons'] for row in valid_ihub_rows)
valid_best_more = len(valid_ihub_rows) - valid_best_fewer - valid_best_same
best_better_than_valid_ihub = sum(objective(row, 'best') < ihub_objective(row) for row in valid_ihub_rows)
best_equal_to_valid_ihub = sum(objective(row, 'best') == ihub_objective(row) for row in valid_ihub_rows)
best_worse_than_valid_ihub = len(valid_ihub_rows) - best_better_than_valid_ihub - best_equal_to_valid_ihub
fast_latency_wins = sum(row['fast_runtime_ms'] < row['ihub_latency_ms'] for row in benchmark_rows)
best_latency_wins = sum(row['best_runtime_ms'] < row['ihub_latency_ms'] for row in benchmark_rows)

display(Markdown('### Executive scorecard'))
display_table([
    {
        'Question': 'Best vs policy-compliant iHub',
        'Evidence': f'{best_better_than_valid_ihub} better / {best_equal_to_valid_ihub} equal / {best_worse_than_valid_ihub} worse',
        'Insight': 'Comparison excludes recorded iHub fill-cap failures',
    },
    {
        'Question': 'Best vs Fast',
        'Evidence': f'{best_improved} improved / {best_tied} tied / {best_worse} worse',
        'Insight': 'Exact search adds quality without degrading the fallback',
    },
    {
        'Question': 'Best proof status',
        'Evidence': f'{best_proven} proven / {order_count - best_proven} time-limited',
        'Insight': 'Timeouts are explicit and return a validated plan',
    },
    {
        'Question': 'Observable iHub fill policy',
        'Evidence': f'{len(valid_ihub_rows)} pass / {box9_over_cap} fail',
        'Insight': 'Policy failures must be separated before comparing quality',
    },
])

display(Markdown('### Why the raw carton comparison is misleading'))
display_table([
    {
        'Population': 'All 2,000 reference outputs',
        'Best fewer / same / more cartons': ' / '.join(map(str, best_counts)),
        'Interpretation': '75 apparent losses are mixed with policy violations',
    },
    {
        'Population': f'{len(valid_ihub_rows)} fill-cap-pass outputs',
        'Best fewer / same / more cartons': f'{valid_best_fewer} / {valid_best_same} / {valid_best_more}',
        'Interpretation': 'After a fair policy filter, higher-carton losses fall to zero',
    },
])

display(Markdown('### Warm latency'))
display_table([
    {
        'System': 'Fast',
        'Median (ms)': f'{statistics.median(fast_times):.3f}',
        'P95 (ms)': f'{percentile_95(fast_times):.3f}',
        'Faster than iHub orders': f'{fast_latency_wins} / {order_count}',
    },
    {
        'System': 'Best',
        'Median (ms)': f'{statistics.median(best_times):.3f}',
        'P95 (ms)': f'{percentile_95(best_times):.3f}',
        'Faster than iHub orders': f'{best_latency_wins} / {order_count}',
    },
    {
        'System': 'iHub reference',
        'Median (ms)': f'{statistics.median(ihub_times):.3f}',
        'P95 (ms)': f'{percentile_95(ihub_times):.3f}',
        'Faster than iHub orders': 'baseline',
    },
])

### Reading the results

- **On par:** counted in the overall evidence, but not worth reviewing one by one.
- **Best wins:** review the order IDs and the reason for improvement; these are the strongest live demo candidates.
- **Best loses:** investigate every policy-compliant loss if any appear.
- **iHub policy exceptions:** inspect separately; a smaller iHub recommendation is not a fair win if it breaks the recorded fill cap.

Select an Order ID from the tables below to explore its live packing decision. The charts are based on the saved benchmark, not a new solver run.

In [ ]:
# Compare Fast and Best against the same policy-compliant iHub reference.
import matplotlib.pyplot as plt
import pandas as pd

benchmark_df = pd.DataFrame(benchmark_rows)
policy_pass = benchmark_df.loc[~benchmark_df['ihub_box9_over_fill_cap']].copy()
policy_exceptions = benchmark_df.loc[benchmark_df['ihub_box9_over_fill_cap']].copy()
objective_fields = ('cartons', 'largest_carton_volume_mm3', 'external_volume_mm3')

def classify(row, mode):
    candidate = tuple(row[f'{mode}_{field}'] for field in objective_fields)
    reference = tuple(row[f'ihub_{field}'] for field in objective_fields)
    return 'Win' if candidate < reference else 'Match' if candidate == reference else 'Lose'

def reason(row, mode, outcome):
    direction = 1 if outcome == 'Lose' else -1
    for field, better, worse in (
        ('cartons', 'Fewer cartons', 'More cartons'),
        ('largest_carton_volume_mm3', 'Smaller largest carton', 'Larger largest carton'),
        ('external_volume_mm3', 'Lower total carton volume', 'Higher total carton volume'),
    ):
        difference = row[f'{mode}_{field}'] - row[f'ihub_{field}']
        if difference * direction > 0:
            return worse if outcome == 'Lose' else better
    raise ValueError('A win or loss must differ on at least one objective')

outcomes = ('Win', 'Match', 'Lose')
modes = ('fast', 'best')
for mode in modes:
    policy_pass[f'{mode}_outcome'] = policy_pass.apply(lambda row: classify(row, mode), axis=1)

# Level 1: show the same Win / Match / Lose categories for both solvers.
summary = pd.DataFrame([
    {'Solver': mode.title(), **{outcome: int((policy_pass[f'{mode}_outcome'] == outcome).sum())
                               for outcome in outcomes}}
    for mode in modes
])
display(summary)
fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(outcomes))
width = 0.35
for index, mode in enumerate(modes):
    values = [int(summary.loc[summary['Solver'] == mode.title(), outcome].iloc[0])
              for outcome in outcomes]
    bars = ax.bar([position + (index - 0.5) * width for position in x],
                  values, width, label=mode.title())
    ax.bar_label(bars, padding=3)
ax.set_xticks(list(x), outcomes)
ax.set_ylim(0, max(1, *summary[list(outcomes)].to_numpy().ravel()) * 1.14)
ax.set_ylabel('Orders')
ax.set_title('Fast and Best vs iHub: policy-compliant orders')
ax.legend()
plt.tight_layout()
plt.show()
display(Markdown(
    f"**Interpretation:** Both solvers are evaluated against the same {len(policy_pass):,} "
    f"policy-compliant iHub orders. The other {len(policy_exceptions):,} iHub fill-policy "
    "exceptions are excluded from both comparisons."
))

# Level 2: break down wins and losses symmetrically for Fast and Best.
win_reasons = ('Fewer cartons', 'Smaller largest carton', 'Lower total carton volume')
loss_reasons = ('More cartons', 'Larger largest carton', 'Higher total carton volume')
categories = []
for mode in modes:
    for outcome in outcomes:
        subset = policy_pass.loc[policy_pass[f'{mode}_outcome'] == outcome].copy()
        if outcome == 'Match':
            categories.append((f'{mode.title()}: Match', subset))
            continue
        subset['reason'] = subset.apply(lambda row: reason(row, mode, outcome), axis=1)
        for label in (win_reasons if outcome == 'Win' else loss_reasons):
            categories.append((f'{mode.title()}: {outcome} — {label}',
                               subset.loc[subset['reason'] == label].copy()))

fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex='col')
for row_index, mode in enumerate(modes):
    for col_index, (outcome, labels) in enumerate(
        (('Win', win_reasons), ('Lose', loss_reasons))
    ):
        ax = axes[row_index, col_index]
        values = [
            len(frame) for title, frame in categories
            if title.startswith(f'{mode.title()}: {outcome} — ')
        ]
        bars = ax.barh(['Carton count', 'Largest carton', 'Total volume'], values)
        ax.bar_label(bars, padding=3)
        ax.set_xlim(0, max(1, *values) * 1.2)
        ax.set_title(f'{mode.title()} {outcome.lower()}s')
        ax.set_xlabel('Orders')
fig.tight_layout()
plt.show()
display(Markdown(
    "**Interpretation:** Each win or loss is attributed to the first differing "
    "objective: carton count, largest carton volume, then total external volume. "
    "Matches need no further breakdown."
))

# Runtime is the other half of the Fast-versus-Best trade-off.
runtime_summary = pd.DataFrame([
    {'Solver': mode.title(),
     'Median runtime (ms)': policy_pass[f'{mode}_runtime_ms'].median(),
     'P95 runtime (ms)': policy_pass[f'{mode}_runtime_ms'].quantile(0.95)}
    for mode in modes
]).round(2)
display(runtime_summary)
fig, ax = plt.subplots(figsize=(7, 3.5))
x = range(len(modes))
width = 0.35
for index, metric in enumerate(('Median runtime (ms)', 'P95 runtime (ms)')):
    bars = ax.bar([position + (index - 0.5) * width for position in x],
                  runtime_summary[metric], width, label=metric)
    ax.bar_label(bars, fmt='%.2f', padding=3)
ax.set_xticks(list(x), ['Fast', 'Best'])
ax.set_ylabel('Solver runtime (ms)')
ax.set_title('Packing quality versus computation time')
ax.legend()
plt.tight_layout()
plt.show()

# Optional order-level inspection — uncomment only what you need.
# Every category retains its full DataFrame in `categories`; the benchmark charts above
# run by default without printing thousands of rows.
#
# display(policy_pass)        # All policy-compliant orders with Fast/Best outcomes.
# display(policy_exceptions)  # iHub orders excluded because of fill-policy exceptions.
#
# Available category names:
# Fast: Win — Fewer cartons / Smaller largest carton / Lower total carton volume
# Fast: Match
# Fast: Lose — More cartons / Larger largest carton / Higher total carton volume
# Best: Win — Fewer cartons / Smaller largest carton / Lower total carton volume
# Best: Match
# Best: Lose — More cartons / Larger largest carton / Higher total carton volume
#
# display(pd.DataFrame([{'Category': title, 'Orders': len(frame)}
#                       for title, frame in categories]))  # Counts by category.
# inspect_category = 'Best: Win — Fewer cartons'  # Change to any category above.
# display(next(frame for title, frame in categories if title == inspect_category))
# display(policy_pass.loc[policy_pass['order_id'] == 428])  # Inspect one order.


## 6. Compare any order with iHub

**This is the main demo.** Change `ORDER_ID` in the next cell to any ID in the sample dataset. The helper runs both solver modes live, using the selected order's own items, candidate cartons and constraints. iHub is shown only as the recorded reference.

A lower carton count wins first; for ties, compare the largest carton, then total carton volume. iHub's recorded fill percentage is checked, but its 3D placement cannot be independently verified from this dataset.

In [ ]:
def compare_order(order_id, *, show_items=False, visualize=False):
    # Run the actual solver for one selected order; iHub is reference only.
    if order_id not in ihub_records:
        raise ValueError(f'Unknown Order ID {order_id}. Choose an ID in ihub_records.')
    record = ihub_records[order_id]
    source = record['input']
    items = [dict(item) for item in source['Items']['ItemsList']]
    boxes_for_order = [dict(box) for box in source['Bins']['BinsList']]
    order = {
        'OrderId': source['OrderId'],
        'OrderNo': source['OrderNo'],
        'Items': items,
    }

    # Read this order's packing policy rather than hard-coding the demo defaults.
    parameters = source['Bins']['Parameters']
    buffer = parameters['BinBuffer']
    threshold = parameters['BinMaxFillCheckMinItemQty']
    high_count_fill = parameters['BinMaxFillPct']
    physical_item_count = sum(item['Quantity'] for item in items)
    fill_cap = high_count_fill if physical_item_count > threshold else 100
    buffer_by_axis = {
        'length': buffer['Length'],
        'width': buffer['Width'],
        'height': buffer['Height'],
    }

    def run(mode):
        return solve_order(
            order,
            boxes_for_order,
            mode=mode,
            high_item_count_threshold=threshold,
            high_item_count_max_fill_pct=high_count_fill,
            max_fill_pct=100,
            bin_buffer=buffer_by_axis,
        )

    # Execute Fast and Best on exactly the same inputs.
    fast = run('fast')
    best = run('best')
    box_by_code = {box['Code']: box for box in boxes_for_order}
    ihub_packed = record['output']['Data']['BinsPacked']

    def external_volume(code):
        box = box_by_code[code]
        return box['Length'] * box['Width'] * box['Height']

    rows = []
    for system, result in [('Fast', fast), ('Best', best)]:
        fills = [round(metric.usable_utilization_pct, 2) for metric in result.metrics.boxes]
        codes = [carton.box_code for carton in result.packed_boxes]
        rows.append({
            'System': system,
            'Rules': 'PASS — validated',
            'Proof': 'PROVEN' if result.optimality_proven else result.search_status.upper(),
            'Cartons': result.metrics.box_count,
            'Boxes': ', '.join(codes),
            'Fill / cap (%)': f"{', '.join(map(str, fills))} / {fill_cap}",
            'Carton volume (mm³)': f'{result.metrics.total_external_box_volume:,.0f}',
            'Warm ms': round(result.runtime_ms, 3),
        })

    # Compare with the stored iHub answer; do not pass it into solve_order().
    ihub_codes = [carton['Code'] for carton in ihub_packed]
    ihub_fills = [round(carton['UsedSpace'], 2) for carton in ihub_packed]
    ihub_fill_pass = all(fill <= fill_cap for fill in ihub_fills)
    rows.append({
        'System': 'iHub',
        'Rules': 'PASS — fill only' if ihub_fill_pass else 'FAIL — fill cap',
        'Proof': 'Not observable',
        'Cartons': len(ihub_packed),
        'Boxes': ', '.join(ihub_codes),
        'Fill / cap (%)': f"{', '.join(map(str, ihub_fills))} / {fill_cap}",
        'Carton volume (mm³)': f"{sum(external_volume(code) for code in ihub_codes):,.0f}",
        'Warm ms': record['latency_ms'],
    })

    # Show source items or the Best 3D placement only when requested.
    if show_items:
        show_details('Original order items', make_table(items))
    display_table(rows)
    if visualize:
        if not best.validation.valid:
            raise AssertionError(best.validation.errors)
        summary, placements = result_tables(best)
        display(summary)
        show_details('Best XYZ placements', placements)
        visualize_result(best)
    return {
        'fast': fast, 'best': best, 'ihub': ihub_packed,
        'rows': rows, 'fill_cap': fill_cap,
    }


### Try any order

Change the Order ID in the next cell to inspect a specific order. The examples below are optional worked scenarios.

In [ ]:
# Change the Order ID to compare another order.
compare_order(428, visualize=True)


### 5.1 Can exact search repair a greedy miss? Yes—order 428

**Live check:** Run the following cell and interpret its current carton count, carton types, validation and runtime. Earlier fixed claims may no longer hold after the Fast update.needs Box4 + Box8; Best proves that one Box8 is feasible and matches iHub. This is the clearest evidence that exact search fixes a real geometric failure rather than merely choosing a different valid layout.

In [ ]:
comparison_428 = compare_order(428)

### 5.2 Can a tie still save packaging? Yes—order 1775

**Live check:** Run the following cell and interpret its current carton count, carton types, validation and runtime. Earlier fixed claims may no longer hold after the Fast update.ne carton, but our validated Box2 has **60.2% less external volume** than iHub's Box4. Carton count alone would hide this win.

In [ ]:
comparison_1775 = compare_order(1775)

### 5.3 Can we use fewer cartons without breaking the cap? Yes—order 321

**Live check:** Run the following cell and interpret its current carton count, carton types, validation and runtime. Earlier fixed claims may no longer hold after the Fast update.nd Best use one Box9 at **68.9% fill**, below the configured 70% cap. This is a valid one-carton business win, not a policy shortcut.

In [ ]:
comparison_321 = compare_order(321)

### 5.4 What if iHub's smaller answer breaks policy? Reject the comparison—order 70

**Live check:** Run the following cell and interpret its current carton count, carton types, validation and runtime. Earlier fixed claims may no longer hold after the Fast update.ne Box9 reports **74.61% fill against a 70% cap**. Best uses two compliant cartons. Counting this as a Best loss would reward a policy violation, so the benchmark separates these cases.

In [ ]:
comparison_70 = compare_order(70)
display_table([
    {'Metric': 'iHub Box9 reported utilization', 'Value': f"{comparison_70['ihub'][0]['UsedSpace']:.1f}%"},
    {'Metric': 'Configured maximum fill above six items', 'Value': '70.0%'},
])

### 5.5 What happens when proof time runs out? Safe fallback—order 1455

**Live check:** Run the following cell and interpret its current carton count, carton types, validation and runtime. Earlier fixed claims may no longer hold after the Fast update.ne Box5. Best cannot finish the proof within the budget, so it returns Fast's validated Box5 and reports `TIME_LIMIT`. The answer remains safe; only the optimality claim is withheld.

In [ ]:
comparison_1455 = compare_order(1455)

## 7. Check for reference-answer leakage

The solver should depend on order items, candidate cartons, and constraints, not on the stored iHub recommendation. The following test deliberately changes the reference answer and verifies that solver outputs remain unchanged.

In [ ]:
from benchmark_solver_modes import benchmark
audit_original = copy.deepcopy(ihub_records[428])
audit_changed = copy.deepcopy(audit_original)
audit_changed['output']['Data']['BinsPacked'] = [{'Code': 'Box9', 'UsedSpace': 1.0}]
audit_changed['latency_ms'] = 999_999

audit_before = benchmark([audit_original])[0]
audit_after = benchmark([audit_changed])[0]
solver_fields = (
    'fast_boxes', 'fast_cartons', 'fast_external_volume_mm3',
    'best_boxes', 'best_cartons', 'best_external_volume_mm3',
)
assert {field: audit_before[field] for field in solver_fields} == {
    field: audit_after[field] for field in solver_fields
}
assert audit_before['ihub_boxes'] != audit_after['ihub_boxes']

display_table([
    {
        'Reference record': 'Original',
        'Stored iHub answer': audit_before['ihub_boxes'],
        'Fast recommendation': audit_before['fast_boxes'],
        'Best recommendation': audit_before['best_boxes'],
    },
    {
        'Reference record': 'iHub answer deliberately changed',
        'Stored iHub answer': audit_after['ihub_boxes'],
        'Fast recommendation': audit_after['fast_boxes'],
        'Best recommendation': audit_after['best_boxes'],
    },
])
display(Markdown('**PASS:** changing the stored iHub answer changes the comparison label, but not either solver recommendation.'))

### Evaluation leakage risk: not yet eliminated

The audit proves there is no **per-order answer leakage** in the runtime path. It does not make the 2,000 orders a pristine holdout: this same dataset informed debugging, objective design, and example selection. That creates development-set selection risk even though the solver is not trained.

A defensible final evaluation should therefore freeze the solver and configuration, run on a new masked holdout whose iHub outputs remain hidden until all plans are saved, reveal the references only for scoring, report every order rather than selected examples, and measure both systems under comparable latency conditions. Until then, the correct claim is **“no direct reference-answer leakage; strong development-benchmark evidence; independent holdout still required.”**

## 8. Recommendation and limitations

**Best** is the preferred quality-oriented mode when its bounded search time is acceptable. **Fast** is appropriate when predictable low latency is more important. Both must respect the same packing constraints.

This development dataset was also used during design and debugging. Do not treat its benchmark as independent proof of generalization. For final evaluation, freeze the solver and configuration before scoring a new masked holdout dataset.

Inspect the benchmark tables above for actual carton counts, policy exceptions, and measured latency rather than relying on fixed numbers copied from an earlier run.